# 03 — Data Cleaning and Feature Engineering

This notebook prepares the Idealista dataset for machine learning.

Main goals:
- Load the EDA-enriched or raw dataset
- Clean duplicates, missing values, outliers and data types
- Create useful real estate features
- Prepare a model-ready dataset
- Export cleaned datasets to `data/processed/`
- Generate a compact Assignment 2 report


## 1. Imports and project paths


In [1]:
from pathlib import Path
from datetime import datetime
import json
import re

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 140)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

PROJECT_ROOT = Path("..") if Path.cwd().name == "notebooks" else Path(".")
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT.resolve())
print("Processed data directory:", PROCESSED_DATA_DIR.resolve())


Project root: C:\Users\hugol\Desktop\Python\B2\ml-poc-project
Processed data directory: C:\Users\hugol\Desktop\Python\B2\ml-poc-project\data\processed


## 2. Load latest available dataset


In [2]:
def find_latest_file(patterns, directories):
    files = []
    for directory in directories:
        for pattern in patterns:
            files.extend(directory.glob(pattern))
    files = sorted(files)
    if not files:
        return None
    return files[-1]

DATA_PATH = find_latest_file(
    patterns=[
        "idealista_eda_enriched_*.csv",
        "idealista_raw_listings_*.csv"
    ],
    directories=[PROCESSED_DATA_DIR, RAW_DATA_DIR]
)

if DATA_PATH is None:
    raise FileNotFoundError(
        "No dataset found. Run 01_data_collection.ipynb and 02_eda.ipynb first."
    )

print("Loading:", DATA_PATH)
df_raw = pd.read_csv(DATA_PATH)
print("Raw shape:", df_raw.shape)
display(df_raw.head())


Loading: ..\data\raw\idealista_raw_listings_20260430_185730.csv
Raw shape: (500, 51)


,propertyCode,thumbnail,externalReference,numPhotos,price,propertyType,operation,size,rooms,bathrooms,address,province,municipality,district,country,neighborhood,latitude,longitude,showAddress,url,distance,description,hasVideo,status,newDevelopment,priceByArea,hasPlan,has3DTour,has360,hasStaging,notes,topNewDevelopment,newDevelopmentHighlight,topPlus,priceInfo.price.amount,priceInfo.price.currencySuffix,parkingSpace.hasParkingSpace,parkingSpace.isParkingSpaceIncludedInPrice,detailedType.typology,detailedType.subTypology,suggestedTexts.subtitle,suggestedTexts.title,highlight.groupDescription,floor,exterior,hasLift,priceInfo.price.priceDropInfo.formerPrice,priceInfo.price.priceDropInfo.priceDropValue,priceInfo.price.priceDropInfo.priceDropPercentage,parkingSpace.parkingSpacePrice,newDevelopmentFinished
0,111308260,https://img4.idealista.com/blur/480_360_mq/0/i...,CA219287,62,"3,950,000.00",chalet,sale,603.00,7,7,calle Guarda,Madrid,Pozuelo de Alarcón,Somosaguas,es,Somosaguas,40.43,-3.78,False,https://www.idealista.com/inmueble/111308260/,6587,"En la exclusiva urbanización El Montecillo, en...",True,good,False,"6,551.00",True,True,False,False,[],False,False,True,"3,950,000.00",€,True,True,chalet,independantHouse,"Somosaguas, Pozuelo de Alarcón",Casa independiente en calle Guarda,Top+,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,110715434,https://img4.idealista.com/blur/480_360_mq/0/i...,EC216671,46,"1,400,000.00",penthouse,sale,177.00,2,2,calle de Isabel la Católica,Madrid,Madrid,Centro,es,Palacio,40.42,-3.71,False,https://www.idealista.com/inmueble/110715434/,617,GILMAR Consulting Inmobiliario pone a su dispo...,True,good,False,"7,910.00",True,True,False,True,[],False,False,True,"1,400,000.00",€,NaN,NaN,flat,penthouse,"Palacio, Madrid",Ático en calle de Isabel la Católica,Top+,6,True,True,"1,500,000.00","100,000.00",7.00,NaN,NaN
2,110921969,https://img4.idealista.com/blur/480_360_mq/0/i...,NY-219592,71,"3,800,000.00",chalet,sale,"1,000.00",7,6,Barrio Somosaguas,Madrid,Pozuelo de Alarcón,Somosaguas,es,Somosaguas,40.43,-3.79,False,https://www.idealista.com/inmueble/110921969/,7775,SIN HONORARIOS A LA PARTE COMPRADORA GILMAR se...,True,renew,False,"3,800.00",True,True,False,True,[],False,False,True,"3,800,000.00",€,True,True,chalet,independantHouse,"Somosaguas, Pozuelo de Alarcón",Casa independiente,Top+,NaN,NaN,NaN,"4,200,000.00","400,000.00",10.00,NaN,NaN
3,109999723,https://img4.idealista.com/blur/480_360_mq/0/i...,DS-P22798,33,"2,100,000.00",flat,sale,137.00,3,3,calle Alcalá,Madrid,Madrid,Barrio de Salamanca,es,Goya,40.42,-3.68,False,https://www.idealista.com/inmueble/109999723/,2404,DIZA Consultores les presenta este fantástico ...,True,good,False,"15,328.00",True,True,False,True,[],False,False,True,"2,100,000.00",€,NaN,NaN,flat,NaN,"Goya, Madrid",Piso en calle Alcalá,Top+,3,True,True,"2,300,000.00","200,000.00",9.00,NaN,NaN
4,103883825,https://img4.idealista.com/blur/480_360_mq/0/i...,AS192516,51,"4,950,000.00",flat,sale,383.00,5,3,calle Maldonado,Madrid,Madrid,Barrio de Salamanca,es,Castellana,40.43,-3.68,False,https://www.idealista.com/inmueble/103883825/,2523,En el corazón del prestigioso barrio de Salama...,True,good,False,"12,924.00",True,True,False,True,[],False,False,True,"4,950,000.00",€,NaN,NaN,flat,NaN,"Castellana, Madrid",Piso en calle Maldonado,Top+,6,True,True,NaN,NaN,NaN,NaN,NaN


## 3. Utility functions


In [3]:
def normalize_column_names(df):
    """Normalize column names for easier modeling."""
    out = df.copy()
    out.columns = [
        col.strip()
           .replace(".", "_")
           .replace(" ", "_")
           .replace("-", "_")
        for col in out.columns
    ]
    return out


def to_bool_series(series):
    """Convert mixed boolean/string/numeric values into nullable boolean."""
    if series is None:
        return None
    true_values = {"true", "yes", "y", "1", "si", "sí"}
    false_values = {"false", "no", "n", "0"}
    def convert(x):
        if pd.isna(x):
            return np.nan
        if isinstance(x, bool):
            return x
        if isinstance(x, (int, float)) and not pd.isna(x):
            if x == 1:
                return True
            if x == 0:
                return False
        s = str(x).strip().lower()
        if s in true_values:
            return True
        if s in false_values:
            return False
        return np.nan
    return series.apply(convert)


def parse_floor_value(x):
    """
    Convert Idealista floor values into an approximate numeric floor.
    bj / bajo -> 0; ss / sotano -> -1; en / entreplanta -> 0.5
    """
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    if s in {"bj", "bajo", "pb", "principal"}:
        return 0.0
    if s in {"ss", "sotano", "sótano", "semisotano", "semisótano"}:
        return -1.0
    if s in {"en", "entreplanta"}:
        return 0.5
    match = re.search(r"-?\d+", s)
    if match:
        return float(match.group(0))
    return np.nan


def safe_divide(a, b):
    return np.where((b != 0) & (~pd.isna(b)), a / b, np.nan)


def find_first_existing_col(df, candidates):
    for col in candidates:
        if col in df.columns:
            return col
    return None


## 4. Initial cleaning


In [4]:
df = df_raw.copy()
initial_rows, initial_cols = df.shape

# Normalize column names because Idealista nested fields contain dots
df = normalize_column_names(df)

print("Initial shape:", (initial_rows, initial_cols))
print("Shape after column normalization:", df.shape)

# Deduplicate by propertyCode if available
duplicate_count = 0
if "propertyCode" in df.columns:
    duplicate_count = int(df["propertyCode"].duplicated().sum())
    df = df.drop_duplicates(subset=["propertyCode"], keep="first")

print("Duplicate propertyCode removed:", duplicate_count)
print("Shape after duplicate removal:", df.shape)


Initial shape: (500, 51)
Shape after column normalization: (500, 51)
Duplicate propertyCode removed: 0
Shape after duplicate removal: (500, 51)


## 5. Data type conversion


In [5]:
numeric_cols = [
    "price", "size", "rooms", "bathrooms", "latitude", "longitude",
    "distance", "numPhotos", "price_per_m2"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

boolean_source_cols = [
    "hasLift",
    "exterior",
    "parkingSpace_hasParkingSpace",
    "parkingSpace_isParkingSpaceIncludedInPrice",
    "newDevelopmentFinished",
    "showAddress"
]

for col in boolean_source_cols:
    if col in df.columns:
        df[col] = to_bool_series(df[col])

if "floor" in df.columns:
    df["floor_numeric"] = df["floor"].apply(parse_floor_value)
    df["floor_missing"] = df["floor_numeric"].isna().astype(int)

print("Data types converted.")
display(df[[col for col in ["price", "size", "rooms", "bathrooms", "floor", "floor_numeric"] if col in df.columns]].head())


Data types converted.


,price,size,rooms,bathrooms,floor,floor_numeric
0,"3,950,000.00",603.00,7,7,NaN,NaN
1,"1,400,000.00",177.00,2,2,6,6.00
2,"3,800,000.00","1,000.00",7,6,NaN,NaN
3,"2,100,000.00",137.00,3,3,3,3.00
4,"4,950,000.00",383.00,5,3,6,6.00


## 6. Missing values strategy


In [6]:
missing_report = (
    pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_pct": df.isna().mean() * 100
    })
    .sort_values("missing_pct", ascending=False)
)

display(missing_report.head(30))

high_missing_threshold = 80
high_missing_cols = missing_report[missing_report["missing_pct"] >= high_missing_threshold].index.tolist()

print("Columns with >= 80% missing values:")
print(high_missing_cols)


,missing_count,missing_pct
newDevelopmentFinished,495,99.00
parkingSpace_parkingSpacePrice,483,96.60
priceInfo_price_priceDropInfo_formerPrice,411,82.20
priceInfo_price_priceDropInfo_priceDropValue,411,82.20
priceInfo_price_priceDropInfo_priceDropPercentage,411,82.20
detailedType_subTypology,340,68.00
parkingSpace_hasParkingSpace,259,51.80
parkingSpace_isParkingSpaceIncludedInPrice,259,51.80
floor,100,20.00
floor_numeric,100,20.00


Columns with >= 80% missing values:
['newDevelopmentFinished', 'parkingSpace_parkingSpacePrice', 'priceInfo_price_priceDropInfo_formerPrice', 'priceInfo_price_priceDropInfo_priceDropValue', 'priceInfo_price_priceDropInfo_priceDropPercentage']


### Missing value decisions

For the model-ready dataset:
- Extremely sparse columns are excluded from the feature set.
- Main numerical columns are kept only when valid.
- Boolean amenities are converted into 0/1 flags with additional missingness indicators where useful.
- Categorical location fields are filled with `"Unknown"`.


In [7]:
bool_mapping = {
    "hasLift": "has_lift",
    "exterior": "is_exterior",
    "parkingSpace_hasParkingSpace": "has_parking",
    "parkingSpace_isParkingSpaceIncludedInPrice": "parking_included",
    "newDevelopmentFinished": "new_development_finished"
}

for source_col, new_col in bool_mapping.items():
    if source_col in df.columns:
        df[f"{new_col}_missing"] = df[source_col].isna().astype(int)
        df[new_col] = df[source_col].fillna(False).astype(bool).astype(int)

price_drop_cols = [
    "priceInfo_price_priceDropInfo_formerPrice",
    "priceInfo_price_priceDropInfo_priceDropValue",
    "priceInfo_price_priceDropInfo_priceDropPercentage"
]
available_price_drop_cols = [col for col in price_drop_cols if col in df.columns]

if available_price_drop_cols:
    df["has_price_drop"] = df[available_price_drop_cols].notna().any(axis=1).astype(int)
else:
    df["has_price_drop"] = 0

categorical_fill_cols = [
    "propertyType", "operation", "province", "municipality",
    "district", "neighborhood", "detailedType_typology", "detailedType_subTypology"
]

for col in categorical_fill_cols:
    if col in df.columns:
        df[col] = df[col].fillna("Unknown").astype(str)

print("Missing value transformations completed.")


Missing value transformations completed.


C:\Users\hugol\AppData\Local\Temp\ipykernel_22048\2021399186.py:12: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[new_col] = df[source_col].fillna(False).astype(bool).astype(int)
C:\Users\hugol\AppData\Local\Temp\ipykernel_22048\2021399186.py:12: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[new_col] = df[source_col].fillna(False).astype(bool).astype(int)
C:\Users\hugol\AppData\Local\Temp\ipykernel_22048\2021399186.py:12: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Cal

## 7. Feature engineering


In [8]:
if "price" in df.columns and "size" in df.columns:
    df["price_per_m2"] = safe_divide(df["price"], df["size"])

if "rooms" in df.columns and "size" in df.columns:
    df["rooms_per_100m2"] = safe_divide(df["rooms"], df["size"]) * 100

if "bathrooms" in df.columns and "rooms" in df.columns:
    df["bathrooms_per_room"] = safe_divide(df["bathrooms"], df["rooms"].replace(0, np.nan))

if "size" in df.columns:
    df["log_size"] = np.log1p(df["size"])

if "price" in df.columns:
    df["log_price"] = np.log1p(df["price"])

if "numPhotos" in df.columns:
    df["log_num_photos"] = np.log1p(df["numPhotos"])

# Location aggregates for exploratory opportunity scoring.
# Important: these use target-derived information and should be treated carefully in modeling.
if "district" in df.columns and "price_per_m2" in df.columns:
    df["district_median_price_per_m2"] = df.groupby("district")["price_per_m2"].transform("median")
    df["district_listing_count"] = df.groupby("district")["price_per_m2"].transform("count")
    df["discount_vs_district_median_pct"] = (
        (df["price_per_m2"] - df["district_median_price_per_m2"])
        / df["district_median_price_per_m2"]
        * 100
    )

if "neighborhood" in df.columns and "price_per_m2" in df.columns:
    df["neighborhood_median_price_per_m2"] = df.groupby("neighborhood")["price_per_m2"].transform("median")
    df["neighborhood_listing_count"] = df.groupby("neighborhood")["price_per_m2"].transform("count")

description_col = find_first_existing_col(
    df,
    ["description", "propertyComment", "suggestedTexts_subtitle", "suggestedTexts_title"]
)

if description_col:
    text = df[description_col].fillna("").astype(str).str.lower()
    df["description_length"] = text.str.len()
    luxury_keywords = ["lujo", "exclusivo", "premium", "alto standing", "vistas", "ático", "piscina", "spa", "domótica", "diseño"]
    renovation_keywords = ["reformado", "a reformar", "reforma", "renovado", "rehabilitado"]
    investment_keywords = ["oportunidad", "inversión", "rentabilidad", "inversor"]
    df["has_luxury_keywords"] = text.apply(lambda s: int(any(k in s for k in luxury_keywords)))
    df["has_renovation_keywords"] = text.apply(lambda s: int(any(k in s for k in renovation_keywords)))
    df["has_investment_keywords"] = text.apply(lambda s: int(any(k in s for k in investment_keywords)))
else:
    df["description_length"] = 0
    df["has_luxury_keywords"] = 0
    df["has_renovation_keywords"] = 0
    df["has_investment_keywords"] = 0

print("Feature engineering completed.")
print("Current shape:", df.shape)

engineered_cols = [
    "price_per_m2", "rooms_per_100m2", "bathrooms_per_room",
    "log_size", "log_price", "log_num_photos",
    "district_median_price_per_m2", "discount_vs_district_median_pct",
    "description_length", "has_luxury_keywords",
    "has_renovation_keywords", "has_investment_keywords"
]
display(df[[col for col in engineered_cols if col in df.columns]].head())


Feature engineering completed.
Current shape: (500, 79)


,price_per_m2,rooms_per_100m2,bathrooms_per_room,log_size,log_price,log_num_photos,district_median_price_per_m2,discount_vs_district_median_pct,description_length,has_luxury_keywords,has_renovation_keywords,has_investment_keywords
0,"6,550.58",1.16,1.00,6.40,15.19,4.14,"5,036.23",30.07,2967,1,0,0
1,"7,909.60",1.13,1.00,5.18,14.15,3.85,"8,074.07",-2.04,2496,1,1,0
2,"3,800.00",0.70,0.86,6.91,15.15,4.28,"5,036.23",-24.55,1746,1,0,0
3,"15,328.47",2.19,1.00,4.93,14.56,3.53,"12,878.07",19.03,1272,1,0,0
4,"12,924.28",1.31,0.60,5.95,15.41,3.95,"12,878.07",0.36,2018,1,0,0


## 8. Outlier treatment


In [10]:
df_before_outliers = df.copy()

filters = pd.Series(True, index=df.index)

if "price" in df.columns:
    filters &= df["price"].between(100_000, 10_000_000)

if "size" in df.columns:
    filters &= df["size"].between(20, 1_000)

if "rooms" in df.columns:
    filters &= df["rooms"].between(0, 10)

if "bathrooms" in df.columns:
    filters &= df["bathrooms"].between(1, 10)

if "price_per_m2" in df.columns:
    filters &= df["price_per_m2"].between(1_000, 30_000)

df_clean = df[filters].copy()

outliers_removed = len(df) - len(df_clean)

print("Rows before outlier filtering:", len(df))
print("Rows after outlier filtering:", len(df_clean))
print("Rows removed:", outliers_removed)
print("Percentage removed:", round(outliers_removed / len(df) * 100, 2), "%")

removed_examples = df.loc[~filters, [col for col in ["propertyCode", "price", "size", "rooms", "bathrooms", "price_per_m2", "district", "url"] if col in df.columns]]
display(removed_examples.head(20))


Rows before outlier filtering: 500
Rows after outlier filtering: 489
Rows removed: 11
Percentage removed: 2.2 %


,propertyCode,price,size,rooms,bathrooms,price_per_m2,district,url
29,106751275,"9,000,000.00","1,674.00",9,10,"5,376.34",Moncloa,https://www.idealista.com/inmueble/106751275/
33,107346813,"9,000,000.00","1,674.00",9,10,"5,376.34",Moncloa,https://www.idealista.com/inmueble/107346813/
98,106269713,"10,450,000.00","1,325.00",6,9,"7,886.79",Somosaguas,https://www.idealista.com/inmueble/106269713/
113,111119262,"14,500,000.00","1,341.00",7,11,"10,812.83",Zona Prado de Somosaguas - La Finca,https://www.idealista.com/inmueble/111119262/
154,108734495,"7,000,000.00","1,035.00",9,9,"6,763.29",Hortaleza,https://www.idealista.com/inmueble/108734495/
155,101541026,"5,000,000.00","1,700.00",4,6,"2,941.18",Moncloa,https://www.idealista.com/inmueble/101541026/
157,111271865,"6,950,000.00","1,380.00",6,6,"5,036.23",Somosaguas,https://www.idealista.com/inmueble/111271865/
159,111001515,"14,500,000.00","1,345.00",7,7,"10,780.67",Zona Prado de Somosaguas - La Finca,https://www.idealista.com/inmueble/111001515/
244,110521316,"14,200,000.00","1,200.00",7,7,"11,833.33",La Moraleja urbanización,https://www.idealista.com/obra-nueva/110521316/
412,108684614,"15,000,000.00","1,500.00",5,6,"10,000.00",Chamartín,https://www.idealista.com/inmueble/108684614/


## 9. Model-ready feature set

Important modeling decision:

`price_per_m2`, `district_median_price_per_m2`, and `discount_vs_district_median_pct` are useful for analysis and investment scoring, but they are target-derived.  
They should not be used directly as standard model inputs for predicting `price`, unless they are computed in a leakage-safe way inside the training pipeline.

For the first modeling dataset, the target will be:

```text
price
```

The input features will exclude direct target-derived variables.


In [11]:
target_col = "price"

candidate_feature_cols = [
    "size", "rooms", "bathrooms", "latitude", "longitude",
    "distance", "numPhotos", "floor_numeric",
    "rooms_per_100m2", "bathrooms_per_room",
    "log_size", "log_num_photos",
    "description_length",
    "has_lift", "has_lift_missing",
    "is_exterior", "is_exterior_missing",
    "has_parking", "has_parking_missing",
    "parking_included", "parking_included_missing",
    "new_development_finished", "new_development_finished_missing",
    "has_price_drop",
    "has_luxury_keywords",
    "has_renovation_keywords",
    "has_investment_keywords",
    "floor_missing",
    "propertyType",
    "district",
    "neighborhood",
    "municipality",
    "detailedType_typology",
    "detailedType_subTypology"
]

feature_cols = [col for col in candidate_feature_cols if col in df_clean.columns]

model_df = df_clean[[target_col] + feature_cols].copy()

essential_cols = [col for col in ["price", "size", "rooms", "bathrooms"] if col in model_df.columns]
model_df = model_df.dropna(subset=essential_cols)

num_features = model_df.select_dtypes(include=[np.number]).columns.tolist()
num_features = [col for col in num_features if col != target_col]
cat_features = model_df.select_dtypes(exclude=[np.number]).columns.tolist()

for col in num_features:
    model_df[col] = model_df[col].fillna(model_df[col].median())

for col in cat_features:
    model_df[col] = model_df[col].fillna("Unknown").astype(str)

print("Model-ready shape:", model_df.shape)
print("Target:", target_col)
print("Number of features:", len(feature_cols))
print("Numerical features:", num_features)
print("Categorical features:", cat_features)

display(model_df.head())


Model-ready shape: (489, 35)
Target: price
Number of features: 34
Numerical features: ['size', 'rooms', 'bathrooms', 'latitude', 'longitude', 'distance', 'numPhotos', 'floor_numeric', 'rooms_per_100m2', 'bathrooms_per_room', 'log_size', 'log_num_photos', 'description_length', 'has_lift', 'has_lift_missing', 'is_exterior', 'is_exterior_missing', 'has_parking', 'has_parking_missing', 'parking_included', 'parking_included_missing', 'new_development_finished', 'new_development_finished_missing', 'has_price_drop', 'has_luxury_keywords', 'has_renovation_keywords', 'has_investment_keywords', 'floor_missing']
Categorical features: ['propertyType', 'district', 'neighborhood', 'municipality', 'detailedType_typology', 'detailedType_subTypology']


,price,size,rooms,bathrooms,latitude,longitude,distance,numPhotos,floor_numeric,rooms_per_100m2,bathrooms_per_room,log_size,log_num_photos,description_length,has_lift,has_lift_missing,is_exterior,is_exterior_missing,has_parking,has_parking_missing,parking_included,parking_included_missing,new_development_finished,new_development_finished_missing,has_price_drop,has_luxury_keywords,has_renovation_keywords,has_investment_keywords,floor_missing,propertyType,district,neighborhood,municipality,detailedType_typology,detailedType_subTypology
0,"3,950,000.00",603.00,7,7,40.43,-3.78,6587,62,3.00,1.16,1.00,6.40,4.14,2967,0,1,0,1,1,0,1,0,0,1,0,1,0,0,1,chalet,Somosaguas,Somosaguas,Pozuelo de Alarcón,chalet,independantHouse
1,"1,400,000.00",177.00,2,2,40.42,-3.71,617,46,6.00,1.13,1.00,5.18,3.85,2496,1,0,1,0,0,1,0,1,0,1,1,1,1,0,0,penthouse,Centro,Palacio,Madrid,flat,penthouse
2,"3,800,000.00","1,000.00",7,6,40.43,-3.79,7775,71,3.00,0.70,0.86,6.91,4.28,1746,0,1,0,1,1,0,1,0,0,1,1,1,0,0,1,chalet,Somosaguas,Somosaguas,Pozuelo de Alarcón,chalet,independantHouse
3,"2,100,000.00",137.00,3,3,40.42,-3.68,2404,33,3.00,2.19,1.00,4.93,3.53,1272,1,0,1,0,0,1,0,1,0,1,1,1,0,0,0,flat,Barrio de Salamanca,Goya,Madrid,flat,Unknown
4,"4,950,000.00",383.00,5,3,40.43,-3.68,2523,51,6.00,1.31,0.60,5.95,3.95,2018,1,0,1,0,0,1,0,1,0,1,0,1,0,0,0,flat,Barrio de Salamanca,Castellana,Madrid,flat,Unknown


## 10. Save cleaned datasets and metadata


In [12]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

clean_full_path = PROCESSED_DATA_DIR / f"idealista_clean_full_{timestamp}.csv"
model_ready_path = PROCESSED_DATA_DIR / f"idealista_model_ready_{timestamp}.csv"
latest_model_ready_path = PROCESSED_DATA_DIR / "idealista_model_ready_latest.csv"
metadata_path = PROCESSED_DATA_DIR / f"idealista_cleaning_metadata_{timestamp}.json"

df_clean.to_csv(clean_full_path, index=False, encoding="utf-8-sig")
model_df.to_csv(model_ready_path, index=False, encoding="utf-8-sig")
model_df.to_csv(latest_model_ready_path, index=False, encoding="utf-8-sig")

metadata = {
    "source_file": str(DATA_PATH),
    "initial_shape": [int(initial_rows), int(initial_cols)],
    "duplicates_removed": int(duplicate_count),
    "rows_after_outlier_filtering": int(len(df_clean)),
    "outliers_removed": int(outliers_removed),
    "model_ready_shape": [int(model_df.shape[0]), int(model_df.shape[1])],
    "target_col": target_col,
    "feature_cols": feature_cols,
    "high_missing_cols_80pct": high_missing_cols,
    "created_at": timestamp
}

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("Saved clean full dataset:", clean_full_path)
print("Saved model-ready dataset:", model_ready_path)
print("Saved latest model-ready dataset:", latest_model_ready_path)
print("Saved metadata:", metadata_path)


Saved clean full dataset: ..\data\processed\idealista_clean_full_20260505_095832.csv
Saved model-ready dataset: ..\data\processed\idealista_model_ready_20260505_095832.csv
Saved latest model-ready dataset: ..\data\processed\idealista_model_ready_latest.csv
Saved metadata: ..\data\processed\idealista_cleaning_metadata_20260505_095832.json
